In [ ]:
import pandas as pd
import pandas_ta as ta
import pyotp
from datetime import datetime, timedelta
import time
import csv
import asyncio
import nest_asyncio
import logging
import websocket as ws_client
import xarray as xr
import threading

from NorenRestApiPy.NorenApi import NorenApi

class ShoonyaApiPy(NorenApi):
    def __init__(self):
        NorenApi.__init__(self, host='https://api.shoonya.com/NorenWClientTP/', websocket='wss://api.shoonya.com/NorenWSTP/')        
        global api
        api = self

import logging
#logging.basicConfig(level=logging.DEBUG)
logging.getLogger('websocket').setLevel(logging.INFO)

usercred = pd.read_excel("usercred.xlsx")
user    = usercred.iloc[0,0]
pwd     = usercred.iloc[0,1]
vc      = usercred.iloc[0,2]
app_key = usercred.iloc[0,3]
imei    = usercred.iloc[0,4]
qr = usercred.iloc[0,5]
factor2 = pyotp.TOTP(qr).now()

api = ShoonyaApiPy()
ret = api.login(userid=user, password=pwd, twoFA=factor2, vendor_code=vc, api_secret=app_key, imei=imei)

In [ ]:

async def trading_logic():
    global last_processed_candle, resampled_data
    while trading_active:
        current_time = pd.Timestamp.now()
        
        for token, data in resampled_data.items():
            if not data.empty:
                last_candle_start = data.index[-1]
                
                # Use a fixed 15-second interval if resample_freq is None
                resample_freq = data.index.freq or pd.Timedelta(seconds=15)
                
                current_candle_end = last_candle_start + resample_freq
                
                if (current_time >= current_candle_end and
                    (token not in last_processed_candle or current_candle_end > last_processed_candle[token])):
                    
                    # Only process if this is truly a new candle
                    if token not in last_processed_candle or current_candle_end > last_processed_candle[token] + resample_freq / 2:
                        last_processed_candle[token] = current_candle_end
                                               
                        
                        process_token_data(token)

        await asyncio.sleep(0.01)  # Check every 10ms


def process_token_data(token):
    df = resampled_data[token]    

    if len(df) > ema_length:
        # Extract the latest and previous EMA values
        EMA_short_latest = df['EMA_short'].iloc[-2]
        EMA_long_latest = df['EMA_long'].iloc[-2]
        ALMA_latest = df['ALMA'].iloc[-2]
        EMA_short_previous = df['EMA_short'].iloc[-3]
        EMA_long_previous = df['EMA_long'].iloc[-3]
        

        timestamp = df.index[-1]
        
        current_position = current_positions.get(token)
        action_taken = False

        # First, check for exit signals
        if current_position == 'long' and (EMA_short_latest < EMA_long_latest):
            log_trade(timestamp, token, "LONG_EXIT", EMA_short_latest, EMA_long_latest, ALMA_latest, EMA_short_previous, EMA_long_previous)
            print(f"Long Exit Signal for {token} at {df.index[-1]} at {pd.Timestamp.now()}")
            print(EMA_short_latest, EMA_long_latest, EMA_short_previous, EMA_long_previous, ALMA_latest)
            current_positions[token] = None
            action_taken = True

        elif current_position == 'short' and (EMA_short_latest > EMA_long_latest):
            log_trade(timestamp, token, "SHORT_EXIT", EMA_short_latest, EMA_long_latest, ALMA_latest, EMA_short_previous, EMA_long_previous)
            print(f"Short Exit Signal for {token} at {df.index[-1]} at {pd.Timestamp.now()}")
            print(EMA_short_latest, EMA_long_latest, EMA_short_previous, EMA_long_previous, ALMA_latest)
            current_positions[token] = None
            action_taken = True

        # Then, check for entry signals if we've just exited a position or if we're not in a position
        if action_taken or current_positions.get(token) is None:
            if (EMA_short_latest > EMA_long_latest >= ALMA_latest) and (EMA_short_previous <= EMA_long_previous):
                log_trade(timestamp, token, "LONG_ENTRY", EMA_short_latest, EMA_long_latest, ALMA_latest, EMA_short_previous, EMA_long_previous)
                print(f"Long Entry Signal for {token} at {df.index[-1]} at {pd.Timestamp.now()}")
                print(EMA_short_latest, EMA_long_latest, EMA_short_previous, EMA_long_previous, ALMA_latest)
                current_positions[token] = 'long'

            elif (EMA_short_latest < EMA_long_latest <= ALMA_latest) and (EMA_short_previous >= EMA_long_previous):
                log_trade(timestamp, token, "SHORT_ENTRY", EMA_short_latest, EMA_long_latest, ALMA_latest, EMA_short_previous, EMA_long_previous)
                print(f"Short Entry Signal for {token} at {df.index[-1]} at {pd.Timestamp.now()}")
                print(EMA_short_latest, EMA_long_latest, EMA_short_previous, EMA_long_previous, ALMA_latest)
                current_positions[token] = 'short'

def log_trade(timestamp, token, action, EMA_short_latest, EMA_long_latest, ALMA_latest, EMA_short_previous, EMA_long_previous):
    with open("trade_log.csv", "a") as log_file:
         log_file.write(f"{timestamp},{token},{action},{EMA_short_latest},{EMA_long_latest},{ALMA_latest},{EMA_short_previous},{EMA_long_previous}\n")     
    

In [ ]:
import asyncio
import threading
from datetime import datetime
import pandas as pd
import nest_asyncio

ema_length = 20
# Constants (you may want to adjust these values)
last_processed_candle = {}
initial_sl_points = 10
move_sl_points = 5
profit_booking_points = 20

trading_active = True
feed_opened = False
feedJson = {}
extra_feedJson = {}
resample_frequency = "15s"
resampling_lock = threading.Lock()
last_resample_time = {}
resampled_data = {}
current_positions = {}

ini = '26009'  ####################################### intial tokens number


if 'resampled_data' not in globals():
    resampled_data = {}
if 'last_resample_time' not in globals():
    last_resample_time = {}

def event_handler_feed_update(tick_data):
    print(tick_data)
    
    try:
        if 'lp' in tick_data and 'tk' in tick_data:
            timest = datetime.fromtimestamp(int(tick_data['ft'])).isoformat()
            token = tick_data['tk']

            with resampling_lock:
                if token not in feedJson:
                    feedJson[token] = []
                    last_resample_time[token] = timest

                feedJson[token].append({'ltp': float(tick_data['lp']), 'tt': timest})

    except (KeyError, ValueError) as e:
        print(f"Error processing tick data: {e}")

async def resample_ticks():
    
    while True:
        if not feedJson:
            await asyncio.sleep(1)
            continue

        with resampling_lock:
            for token, ticks in feedJson.items():
                try:
                    if ticks:
                        # Create a DataFrame with the new ticks
                        df_new = pd.DataFrame(ticks)
                        df_new["tt"] = pd.to_datetime(df_new["tt"])
                        df_new.set_index("tt", inplace=True)

                        # Determine the current resample interval
                        current_resample_interval = df_new.index.max().floor(resample_frequency)

                        # Initialize or update the resampled DataFrame
                        if token not in resampled_data:
                            # Initialize the DataFrame with the first resample
                            df_resampled = df_new['ltp'].resample(resample_frequency).ohlc()
                            resampled_data[token] = df_resampled
                            last_resample_time[token] = df_resampled.index.max()
                        else:
                            # Resample the new ticks
                            df_resampled = df_new['ltp'].resample(resample_frequency).ohlc()

                            # Update the existing resampled DataFrame with new data
                            for idx, row in df_resampled.iterrows():
                                if idx in resampled_data[token].index:
                                    resampled_data[token].loc[idx, 'high'] = max(resampled_data[token].loc[idx, 'high'], row['high'])
                                    resampled_data[token].loc[idx, 'low'] = min(resampled_data[token].loc[idx, 'low'], row['low'])
                                    resampled_data[token].loc[idx, 'close'] = row['close']
                                else:
                                    resampled_data[token].loc[idx] = row

                            # Update the last resampled time
                            last_resample_time[token] = df_resampled.index.max()

                        # Calculate indicators for the updated resampled data
                        # Example: Calculate EMA
                        resampled_data[token]['EMA_short'] = ta.ema(resampled_data[token]['close'], length=3)
                        resampled_data[token]['EMA_long'] = ta.ema(resampled_data[token]['close'], length=10)
                        #resampled_data[token]['ALMA'] = ta.alma(resampled_data[token]['close'], length=20, sigma=6, distribution_offset=0.65)
                        resampled_data[token]['ALMA'] = ta.ema(resampled_data[token]['close'], length=20)
                        #resampled_data[token]['JMA1'] = ta.jma(resampled_data[token]['close'], length=50, phase=0, offset=0)
                        #print(feedJson)
                    feedJson[token] = []

                except Exception as e:
                    if isinstance(e, KeyError) and e.args[0] == 'tt':
                        print(f"Market likely closed for token {token}")
                    else:
                        print(f"Error resampling data for token {token}: {e}")

        await asyncio.sleep(1)
        

def event_handler_order_update(tick_data):
    print(f"Order update {tick_data}")

def open_callback():
    global feed_opened
    feed_opened = True

# Assuming `api` is defined and starts the WebSocket connection
async def connect_and_subscribe():
    global feed_opened
    api.start_websocket(
        order_update_callback=event_handler_order_update,
        subscribe_callback=event_handler_feed_update,
        socket_open_callback=open_callback
    )
    while not feed_opened:
        await asyncio.sleep(1)  # Wait for initial connection
    api.subscribe(['NSE|26009'])
   

async def main():
   
    resample_task = asyncio.create_task(resample_ticks())
    trading_task = asyncio.create_task(trading_logic())
    connect_task = asyncio.create_task(connect_and_subscribe())  
    update_symbols_task = asyncio.create_task(update_atm_symbols())
    
    await asyncio.gather(connect_task, resample_task, trading_task, update_symbols_task)

def set_trading_active(value):
    global trading_active
    trading_active = value
    #print(f"Trading active set to {trading_active}")

# Get the current event loop
loop = asyncio.get_event_loop()

if loop.is_running():
    nest_asyncio.apply()
    asyncio.create_task(main())
else:
    loop.run_until_complete(main())

In [ ]:
def append_to_csv(trade_log):
    with open('trading_log.csv', 'a', newline='') as file:
        writer = csv.writer(file)
        writer.writerow()

import pandas as pd

# Assuming resampled_data has the format {'token_name': DataFrame}
token, df = next(iter(resampled_data.items()))  # Get the token name and DataFrame

df.to_csv(f"{token}.csv", index=True)   # Save the DataFrame to a CSV file (include the index)
print(f"Saved data to {token}.csv")   


In [ ]:
fno_scrips_mcx = pd.read_csv('//home/deep/Desktop/NEWshoonya/MCX_symbols.txt.zip',compression='zip', engine='python',delimiter=',')
fno_scrips_mcx['Expiry'] = pd.to_datetime(fno_scrips_mcx['Expiry'])
fno_scrips_mcx['StrikePrice'] = fno_scrips_mcx['StrikePrice'].astype(float)
fno_scrips_mcx.sort_values('Expiry',inplace=True)
fno_scrips_mcx.reset_index(drop=True, inplace=True)


fno_scrips = pd.read_csv('/home/deep/Desktop/NEWshoonya/NFO_symbols.txt.zip',compression='zip', engine='python',delimiter=',')
fno_scrips['Expiry'] = pd.to_datetime(fno_scrips['Expiry'])
fno_scrips['StrikePrice'] = fno_scrips['StrikePrice'].astype(float)
fno_scrips.sort_values('Expiry',inplace=True)
fno_scrips.reset_index(drop=True, inplace=True)

symbol = 'CRUDEOILM'
async def find_atm_strike_and_symbols():
    global ce_trading_symbol, pe_trading_symbol, ce_trading_token, pe_trading_token
    
    cmp_bn = 6781
    #cmp_bn = float(resampled_data[ini]['close'].iloc[-1])
    # In a real scenario, you might want to fetch this price asynchronously
    # cmp_bn = await get_current_price()  # You'd need to implement this function
    
    mod = int(cmp_bn) % 100
    atm_strike = cmp_bn - mod if mod <= 50 else cmp_bn + (100 - mod)

    # Assuming fno_scrips_mcx is a global DataFrame
    filtered_df = fno_scrips_mcx[
        (fno_scrips_mcx['Symbol'] == symbol) &
        (fno_scrips_mcx['StrikePrice'] == float(atm_strike))
    ]
    
    if filtered_df.empty:
        print(f"Could not find trading symbols for ATM strike {atm_strike}")
        return None, None, None, None

    ce_row = filtered_df[filtered_df['OptionType'] == 'CE'].sort_values('Expiry').iloc[0]
    pe_row = filtered_df[filtered_df['OptionType'] == 'PE'].sort_values('Expiry').iloc[0]

    ce_trading_symbol, pe_trading_symbol = ce_row['TradingSymbol'], pe_row['TradingSymbol']
    ce_trading_token, pe_trading_token = ce_row['Token'], pe_row['Token']
    
    print(f"Updated ATM CE Symbol: {ce_trading_symbol}, Token: {ce_trading_token}")
    print(f"Updated ATM PE Symbol: {pe_trading_symbol}, Token: {pe_trading_token}")

    return ce_trading_symbol, pe_trading_symbol, ce_trading_token, pe_trading_token

async def update_atm_symbols():
    while True:
        await find_atm_strike_and_symbols()
        await asyncio.sleep(15)  # Update every 60 seconds, adjust as needed

In [ ]:
resampled_data